# Actividad MLOps – Semana 2: Pipelines y Calidad

**Proyecto:** clasificación de especies del conjunto de datos Iris mediante Random Forest.

Este notebook adapta la actividad completa al entorno **Jupyter Notebook**. No utiliza `__file__` ni depende de ejecutar un archivo `run_pipeline.py`: la orquestación se realiza mediante funciones definidas y llamadas desde las celdas.


## 1. Configuración del entorno y del proyecto

En Jupyter no se debe utilizar `Path(__file__)`, porque esa variable normalmente no existe en las celdas. En su lugar se utiliza `Path.cwd()` para identificar el directorio de trabajo.

La siguiente celda crea las carpetas necesarias. Si ya estás dentro de la carpeta de tu proyecto, la utilizará; si no, puedes indicar una ruta en `PROJECT_DIR`.

In [18]:
from pathlib import Path
import os
import sys

# Opción A: utilizar la carpeta actual del notebook.
PROJECT_DIR = Path.cwd().resolve()

# Opción B: si necesitas otra ubicación, descomenta y modifica esta línea:
# PROJECT_DIR = Path(r"C:\\Users\\TU_USUARIO\\Documents\\PROYECTO_MLOPS_IRIS")

# Carpetas del proyecto
DATA_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
SRC_DIR = PROJECT_DIR / "src"
TESTS_DIR = PROJECT_DIR / "tests"
CONTRACTS_DIR = PROJECT_DIR / "contracts"

for carpeta in [DATA_DIR, MODELS_DIR, SRC_DIR, TESTS_DIR, CONTRACTS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Fijar el directorio de trabajo para que las rutas relativas sean consistentes.
os.chdir(PROJECT_DIR)

print("Directorio raíz del proyecto:", PROJECT_DIR)
print("\nCarpetas verificadas:")
for carpeta in [DATA_DIR, MODELS_DIR, SRC_DIR, TESTS_DIR, CONTRACTS_DIR]:
    print("-", carpeta.relative_to(PROJECT_DIR))

Directorio raíz del proyecto: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II

Carpetas verificadas:
- data\processed
- models
- src
- tests
- contracts


### Observaciones
- `Path.cwd()` devuelve la carpeta desde la que se está ejecutando el kernel, que puede no coincidir con la ubicación física del notebook.
- Revisa el directorio impreso. Debe ser la raíz donde deseas guardar `data`, `models`, `src`, `tests` y `contracts`.
- Si la ruta no es correcta, modifica `PROJECT_DIR` y vuelve a ejecutar esta celda.

## 2. Dependencias

El proyecto utiliza pandas y scikit-learn para preparar y entrenar el modelo; joblib para guardar el artefacto; MLflow para registrar experimentos; y pytest para las pruebas.

La validación de datos del apartado 4 se implementa con módulos estándar de Python (`csv`, `json` y `math`), sin Great Expectations ni otra librería externa.

In [19]:
# Ejecuta esta celda si alguna dependencia no está instalada.
# En Jupyter, %pip instala en el entorno asociado al kernel.

%pip install pandas scikit-learn joblib mlflow pytest

Note: you may need to restart the kernel to use updated packages.


## 3. Preparación del dataset Iris

Se carga el dataset Iris desde scikit-learn, se conservan los nombres de las cuatro variables predictoras y se agrega la columna objetivo `species`. Luego se divide en entrenamiento (80 %) y prueba (20 %), utilizando `random_state=42` para facilitar la reproducibilidad.

In [20]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

def prepare_data():
    iris = load_iris()
    df = pd.DataFrame(iris.data, columns=iris.feature_names)
    df["species"] = iris.target

    train, test = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df["species"]
    )

    train_path = DATA_DIR / "train.csv"
    test_path = DATA_DIR / "test.csv"

    train.to_csv(train_path, index=False)
    test.to_csv(test_path, index=False)

    print(f"Preparación completada: {len(train)} registros de entrenamiento y {len(test)} de prueba.")
    print("Archivo:", train_path)
    print("Archivo:", test_path)
    return train, test

train_df, test_df = prepare_data()
display(train_df.head())

Preparación completada: 120 registros de entrenamiento y 30 de prueba.
Archivo: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II\data\processed\train.csv
Archivo: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II\data\processed\test.csv


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
8,4.4,2.9,1.4,0.2,0
106,4.9,2.5,4.5,1.7,2
76,6.8,2.8,4.8,1.4,1
9,4.9,3.1,1.5,0.1,0
89,5.5,2.5,4.0,1.3,1


## 4. Validación de calidad de datos sin librerías externas

Se establecen reglas para:
- comprobar que estén presentes las columnas esperadas y en el orden definido;
- detectar campos vacíos;
- comprobar que las cuatro mediciones sean numéricas y finitas;
- comprobar que `species` sea un entero perteneciente a `{0, 1, 2}`;
- verificar rangos lógicos definidos para este ejercicio.

Los rangos son reglas de control adoptadas para el dataset Iris de esta actividad; no deben interpretarse como límites universales para cualquier población de flores.

In [21]:
import csv
import math

FEATURE_RANGES = {
    "sepal length (cm)": (4.0, 8.0),
    "sepal width (cm)": (2.0, 4.5),
    "petal length (cm)": (1.0, 7.0),
    "petal width (cm)": (0.1, 2.6),
}
EXPECTED_COLUMNS = list(FEATURE_RANGES.keys()) + ["species"]
VALID_CLASSES = {0, 1, 2}

def validate_csv(csv_path):
    csv_path = Path(csv_path)
    errors = []

    if not csv_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")

    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)

        if reader.fieldnames != EXPECTED_COLUMNS:
            raise ValueError(
                f"Columnas incorrectas. Esperadas: {EXPECTED_COLUMNS}; "
                f"recibidas: {reader.fieldnames}"
            )

        row_count = 0
        for row_number, row in enumerate(reader, start=2):
            row_count += 1

            for column in EXPECTED_COLUMNS:
                value = row.get(column)

                if value is None or value.strip() == "":
                    errors.append(f"Fila {row_number}: valor vacío en '{column}'.")
                    continue

                if column == "species":
                    try:
                        parsed = int(value)
                        if str(parsed) != value.strip():
                            raise ValueError
                        if parsed not in VALID_CLASSES:
                            errors.append(
                                f"Fila {row_number}: clase '{parsed}' no permitida en species."
                            )
                    except ValueError:
                        errors.append(
                            f"Fila {row_number}: '{column}' debe ser un entero 0, 1 o 2."
                        )
                else:
                    try:
                        parsed = float(value)
                    except ValueError:
                        errors.append(
                            f"Fila {row_number}: '{column}' debe ser numérico."
                        )
                        continue

                    if not math.isfinite(parsed):
                        errors.append(
                            f"Fila {row_number}: '{column}' debe ser un número finito."
                        )
                        continue

                    minimum, maximum = FEATURE_RANGES[column]
                    if not minimum <= parsed <= maximum:
                        errors.append(
                            f"Fila {row_number}: '{column}'={parsed} está fuera "
                            f"del rango [{minimum}, {maximum}]."
                        )

        if row_count == 0:
            errors.append("El archivo no contiene registros.")

    if errors:
        message = "\n".join(errors)
        raise ValueError(f"Validación fallida para {csv_path.name}:\n{message}")

    print(f"VALIDACIÓN EXITOSA: {csv_path.name} ({row_count} registros)")
    return True


def validate_datasets():
    validate_csv(DATA_DIR / "train.csv")
    validate_csv(DATA_DIR / "test.csv")
    print("Todos los conjuntos de datos cumplen las reglas.")

validate_datasets()

VALIDACIÓN EXITOSA: train.csv (120 registros)
VALIDACIÓN EXITOSA: test.csv (30 registros)
Todos los conjuntos de datos cumplen las reglas.


### 4.1. Prueba controlada de la validación

La siguiente celda crea temporalmente un CSV inválido para comprobar que el validador detecta errores. No modifica los archivos oficiales `train.csv` ni `test.csv`.

In [22]:
invalid_test_path = DATA_DIR / "validation_example_invalid.csv"

invalid_row = {
    "sepal length (cm)": "9.5",   # supera el máximo permitido
    "sepal width (cm)": "3.5",
    "petal length (cm)": "",      # valor vacío
    "petal width (cm)": "0.2",
    "species": "1"
}

with invalid_test_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=EXPECTED_COLUMNS)
    writer.writeheader()
    writer.writerow(invalid_row)

try:
    validate_csv(invalid_test_path)
except ValueError as error:
    print("La prueba detectó correctamente los problemas:")
    print(error)
finally:
    invalid_test_path.unlink(missing_ok=True)

La prueba detectó correctamente los problemas:
Validación fallida para validation_example_invalid.csv:
Fila 2: 'sepal length (cm)'=9.5 está fuera del rango [4.0, 8.0].
Fila 2: valor vacío en 'petal length (cm)'.


## 5. Contrato oficial de datos en JSON

El contrato describe las cuatro variables requeridas durante la inferencia. La variable `species` es la etiqueta objetivo conocida durante el entrenamiento y la evaluación; normalmente no se envía como entrada cuando se solicita una predicción.

Se guardará un ejemplo válido y otro inválido para documentar el comportamiento esperado.

In [24]:
import json

data_contract = {
    "contract_name": "Iris Data Contract",
    "version": "1.0.0",
    "project": "PROYECTO_MLOPS_IRIS",
    "description": "Contrato de entrada para inferencia del clasificador Iris.",
    "input_schema": {
        name: {
            "type": "number",
            "required": True,
            "minimum": limits[0],
            "maximum": limits[1]
        }
        for name, limits in FEATURE_RANGES.items()
    },
    "target": {
        "name": "species",
        "type": "integer",
        "allowed_values": [0, 1, 2],
        "description": "Etiqueta usada en entrenamiento y evaluación; no requerida para inferencia."
    },
    "valid_example": {
        "sepal length (cm)": 5.1,
        "sepal width (cm)": 3.5,
        "petal length (cm)": 1.4,
        "petal width (cm)": 0.2
    },
    "invalid_example": {
        "sepal length (cm)": 9.5,
        "sepal width (cm)": 3.5,
        "petal length (cm)": None,
        "petal width (cm)": 0.2
    },
    "validation_rules": [
        "Deben enviarse las cuatro variables de entrada.",
        "No se permiten valores nulos.",
        "Las variables deben ser numéricas y finitas.",
        "Cada variable debe respetar su rango permitido."
    ]
}

contract_path = CONTRACTS_DIR / "data_contract.json"
with contract_path.open("w", encoding="utf-8") as file:
    json.dump(data_contract, file, ensure_ascii=False, indent=4)

print("Contrato creado:", contract_path)
print(json.dumps(data_contract, ensure_ascii=False, indent=4))

Contrato creado: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II\contracts\data_contract.json
{
    "contract_name": "Iris Data Contract",
    "version": "1.0.0",
    "project": "PROYECTO_MLOPS_IRIS",
    "description": "Contrato de entrada para inferencia del clasificador Iris.",
    "input_schema": {
        "sepal length (cm)": {
            "type": "number",
            "required": true,
            "minimum": 4.0,
            "maximum": 8.0
        },
        "sepal width (cm)": {
            "type": "number",
            "required": true,
            "minimum": 2.0,
            "maximum": 4.5
        },
        "petal length (cm)": {
            "type": "number",
            "required": true,
            "minimum": 1.0,
            "maximum": 7.0
        },
        "petal width (cm)": {
            "type": "number",
            "required": true,
            "minimum": 0.1,
            "maximum": 2.6
        }
    },
    "target": {


### 5.1. Validar un registro individual contra el contrato

Esta función aplica las reglas del contrato a una entrada individual, como la que podría recibir una API de inferencia.

In [25]:
def validate_inference_record(record, contract=data_contract):
    schema = contract["input_schema"]
    errors = []

    missing = [name for name, rules in schema.items()
               if rules["required"] and name not in record]
    if missing:
        errors.append(f"Faltan variables obligatorias: {missing}")

    extra = [name for name in record if name not in schema]
    if extra:
        errors.append(f"Variables no reconocidas: {extra}")

    for name, rules in schema.items():
        if name not in record:
            continue

        value = record[name]

        if value is None or isinstance(value, bool):
            errors.append(f"'{name}' no puede ser nulo ni booleano.")
            continue

        if not isinstance(value, (int, float)) or not math.isfinite(value):
            errors.append(f"'{name}' debe ser numérico y finito.")
            continue

        if not rules["minimum"] <= value <= rules["maximum"]:
            errors.append(
                f"'{name}' debe estar entre {rules['minimum']} y {rules['maximum']}."
            )

    return {"valid": len(errors) == 0, "errors": errors}


print("Entrada válida:", validate_inference_record(data_contract["valid_example"]))
print("Entrada inválida:", validate_inference_record(data_contract["invalid_example"]))

Entrada válida: {'valid': True, 'errors': []}
Entrada inválida: {'valid': False, 'errors': ["'sepal length (cm)' debe estar entre 4.0 y 8.0.", "'petal length (cm)' no puede ser nulo ni booleano."]}


**Observaciones:** el JSON es el documento de referencia del contrato. La función anterior es una implementación de sus reglas para una entrada individual. Si se modifica el contrato, deben revisarse también las funciones de validación para mantenerlas sincronizadas.

## 6. Entrenamiento, evaluación y registro con MLflow

Se entrena un `RandomForestClassifier` con 100 árboles y semilla 42. Se evalúa con el conjunto de prueba y se guarda el modelo en `models/model.pkl`.

MLflow registra los parámetros y la exactitud. Si MLflow no está disponible o el registro presenta un problema de configuración, el entrenamiento y el guardado local del modelo siguen siendo posibles; el mensaje lo indicará.

In [26]:

import joblib
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

MODEL_PATH = MODELS_DIR / "model.pkl"


def train_model():

    # ==========================================
    # 1. CARGAR DATOS
    # ==========================================

    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")

    X_train = train[list(FEATURE_RANGES.keys())]
    y_train = train["species"]

    X_test = test[list(FEATURE_RANGES.keys())]
    y_test = test["species"]

    # ==========================================
    # 2. ENTRENAR MODELO
    # ==========================================

    n_estimators = 100

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=42
    )

    model.fit(X_train, y_train)

    # ==========================================
    # 3. EVALUAR MODELO
    # ==========================================

    predictions = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    print(f"Exactitud en prueba: {accuracy:.4f}")

    print("\nReporte de clasificación:")

    print(
        classification_report(
            y_test,
            predictions
        )
    )

    # ==========================================
    # 4. GUARDAR MODELO LOCALMENTE
    # ==========================================

    MODELS_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    joblib.dump(model, MODEL_PATH)

    print("\nModelo guardado correctamente.")

    # ==========================================
    # 5. REGISTRO EN MLFLOW
    # ==========================================

    try:

        mlflow.set_experiment(
            "Proyecto_Final_Grado"
        )

        with mlflow.start_run():

            # Registrar parámetros
            mlflow.log_param(
                "model",
                "RandomForestClassifier"
            )

            mlflow.log_param(
                "n_estimators",
                n_estimators
            )

            mlflow.log_param(
                "random_state",
                42
            )

            # Registrar métricas
            mlflow.log_metric(
                "accuracy",
                float(accuracy)
            )

            # Registrar modelo
            mlflow.sklearn.log_model(
                model,
                name="iris_model",
                skops_trusted_types=[
                    "sklearn.tree._tree.Tree"
                ]
            )

        print("\nExperimento registrado correctamente en MLflow.")

    except Exception as error:

        print("\nADVERTENCIA: no se pudo registrar en MLflow.")

        print("Detalle del error:", error)

        print("El modelo local sí fue guardado correctamente.")

    # ==========================================
    # 6. RESULTADOS FINALES
    # ==========================================

    print("\n" + "=" * 50)
    print("RESUMEN DEL ENTRENAMIENTO")
    print("=" * 50)

    print(f"Algoritmo: Random Forest")
    print(f"Número de árboles: {n_estimators}")
    print(f"Exactitud: {accuracy:.4f}")
    print(f"Modelo guardado: {MODEL_PATH}")

    return model, accuracy


# Ejecutar entrenamiento
model, model_accuracy = train_model()


Exactitud en prueba: 0.9000

Reporte de clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.82      0.90      0.86        10
           2       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30


Modelo guardado correctamente.

Experimento registrado correctamente en MLflow.

RESUMEN DEL ENTRENAMIENTO
Algoritmo: Random Forest
Número de árboles: 100
Exactitud: 0.9000
Modelo guardado: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II\models\model.pkl


**Observaciones:** la exactitud depende de la partición y de la configuración del modelo. No debe afirmarse un valor antes de ejecutar el notebook. El archivo `model.pkl` es el artefacto local que se utilizará en las pruebas.

## 7. Pruebas automáticas con pytest

Se generan pruebas para comprobar que:
- los CSV existen y contienen registros;
- el modelo fue guardado y puede cargarse;
- el modelo supera una exactitud mínima de 0.80 en el conjunto de prueba.

Las pruebas se guardan en `tests/test_model.py` y se ejecutan desde el notebook.

In [27]:
test_file_content = r'''
from pathlib import Path
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score

ROOT = Path.cwd()
DATA = ROOT / "data" / "processed"
MODEL = ROOT / "models" / "model.pkl"
FEATURES = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
]

def test_data_loading():
    train = pd.read_csv(DATA / "train.csv")
    test = pd.read_csv(DATA / "test.csv")
    assert not train.empty
    assert not test.empty
    assert "species" in train.columns

def test_model_exists_and_loads():
    assert MODEL.exists()
    loaded_model = joblib.load(MODEL)
    assert loaded_model is not None

def test_model_accuracy():
    loaded_model = joblib.load(MODEL)
    test = pd.read_csv(DATA / "test.csv")
    predictions = loaded_model.predict(test[FEATURES])
    accuracy = accuracy_score(test["species"], predictions)
    assert accuracy > 0.80, f"Exactitud insuficiente: {accuracy:.3f}"
'''

test_path = TESTS_DIR / "test_model.py"
test_path.write_text(test_file_content, encoding="utf-8")
print("Archivo de pruebas creado:", test_path)

Archivo de pruebas creado: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II\tests\test_model.py


In [28]:
import pytest

# Ejecutar pytest dentro del kernel de Jupyter
pytest_exit_code = pytest.main(["-v", str(TESTS_DIR)])

print("Código de salida de pytest:", pytest_exit_code)
if pytest_exit_code != 0:
    raise RuntimeError("Una o más pruebas automáticas fallaron.")

============================= test session starts =============================
platform win32 -- Python 3.11.17, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\Vivobook\anaconda3\envs\rag_polizas\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II
plugins: platformdirs-4.13.5, anyio-4.15.1
collecting ... collected 3 items

tests/test_model.py::test_data_loading PASSED                            [ 33%]
tests/test_model.py::test_model_exists_and_loads PASSED                  [ 66%]
tests/test_model.py::test_model_accuracy PASSED                          [100%]

============================== warnings summary ===============================
..\..\..\..\..\anaconda3\envs\rag_polizas\Lib\site-packages\_pytest\config\__init__.py:1507
  c:\Users\Vivobook\anaconda3\envs\rag_polizas\Lib\site-packages\_pytest\config\__init__.py:1507: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
 

## 8. Orquestación: ejecutar todo con una sola llamada

En lugar de lanzar procesos externos con `subprocess`, el notebook coordina directamente las funciones definidas en las celdas anteriores. Esto evita problemas con `__file__` y facilita observar los resultados dentro de Jupyter.

Una vez ejecutadas las celdas de definición, el proceso completo se inicia con `run_pipeline()`.

In [29]:
def run_pipeline():
    print("=" * 65)
    print("INICIANDO PIPELINE MLOPS - DATASET IRIS")
    print("=" * 65)

    stages = [
        ("Preparación de datos", prepare_data),
        ("Validación de datos", validate_datasets),
        ("Creación y validación del contrato", lambda: (
            contract_path.exists()
            and validate_inference_record(data_contract["valid_example"])["valid"]
        )),
        ("Entrenamiento del modelo", train_model),
        ("Pruebas automáticas", lambda: pytest.main(["-q", str(TESTS_DIR)]))
    ]

    results = {}

    for stage_name, stage_function in stages:
        print("\n" + "-" * 65)
        print("ETAPA:", stage_name)
        print("-" * 65)

        result = stage_function()

        if stage_name == "Pruebas automáticas" and result != 0:
            raise RuntimeError(f"Falló la etapa: {stage_name}")

        results[stage_name] = "COMPLETADA"
        print("ETAPA COMPLETADA:", stage_name)

    print("\n" + "=" * 65)
    print("PIPELINE EJECUTADO CORRECTAMENTE")
    print("=" * 65)
    return results

# Comando único para ejecutar el flujo completo:
pipeline_results = run_pipeline()
pipeline_results

INICIANDO PIPELINE MLOPS - DATASET IRIS

-----------------------------------------------------------------
ETAPA: Preparación de datos
-----------------------------------------------------------------
Preparación completada: 120 registros de entrenamiento y 30 de prueba.
Archivo: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II\data\processed\train.csv
Archivo: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II\data\processed\test.csv
ETAPA COMPLETADA: Preparación de datos

-----------------------------------------------------------------
ETAPA: Validación de datos
-----------------------------------------------------------------
VALIDACIÓN EXITOSA: train.csv (120 registros)
VALIDACIÓN EXITOSA: test.csv (30 registros)
Todos los conjuntos de datos cumplen las reglas.
ETAPA COMPLETADA: Validación de datos

-----------------------------------------------------------------
ETAPA: Creación y validaci

{'Preparación de datos': 'COMPLETADA',
 'Validación de datos': 'COMPLETADA',
 'Creación y validación del contrato': 'COMPLETADA',
 'Entrenamiento del modelo': 'COMPLETADA',
 'Pruebas automáticas': 'COMPLETADA'}

**Observación importante:** `run_pipeline()` vuelve a preparar los datos y entrenar el modelo. Esto es intencional para que el flujo sea reproducible desde el propio notebook. Las funciones deben haberse definido previamente ejecutando las celdas en orden.

## 9. Documentación del proyecto: generar README.md

El README describe requisitos, estructura, instalación y ejecución. Como el flujo principal está implementado en este notebook, las instrucciones explican cómo abrirlo y ejecutar todas sus celdas. También se documentan los archivos que se generan.

In [30]:
readme_content = '''# PROYECTO MLOPS IRIS

## 1. Descripción
Proyecto académico de clasificación de especies Iris mediante Random Forest.
Incluye preparación de datos, validación de calidad, contrato JSON, entrenamiento,
evaluación, pruebas automatizadas y registro opcional de experimentos en MLflow.

## 2. Tecnologías
- Python 3.10 o superior
- Jupyter Notebook
- pandas
- scikit-learn
- joblib
- MLflow
- pytest

## 3. Estructura
- `data/processed/`: conjuntos train.csv y test.csv.
- `models/`: modelo entrenado model.pkl.
- `contracts/`: contrato de datos data_contract.json.
- `tests/`: pruebas automáticas.
- `src/`: espacio para los scripts equivalentes si se desea migrar el flujo a archivos .py.
- `Actividad_Semana_2_MLOps_Iris.ipynb`: notebook principal.

## 4. Instalación
Crear y activar un entorno virtual:

```bash
python -m venv .venv
```

Windows:
```powershell
.venv\\Scripts\\activate
```

Instalar dependencias:
```bash
python -m pip install --upgrade pip
pip install pandas scikit-learn joblib mlflow pytest jupyter
```

## 5. Ejecución
1. Abrir Jupyter Notebook desde la carpeta raíz del proyecto.
2. Abrir `Actividad_Semana_2_MLOps_Iris.ipynb`.
3. Verificar que `PROJECT_DIR` apunte a la carpeta raíz correcta.
4. Ejecutar las celdas en orden.
5. Al finalizar las definiciones, ejecutar `run_pipeline()` para repetir el pipeline completo.

## 6. Validación
El validador utiliza módulos estándar de Python para verificar columnas, valores vacíos,
tipos numéricos, valores finitos, clases y rangos definidos para Iris.

## 7. Contrato de datos
El archivo `contracts/data_contract.json` documenta las cuatro variables requeridas
para inferencia, sus tipos, rangos y ejemplos de entrada válida e inválida.

## 8. Modelo y métricas
Se utiliza RandomForestClassifier con 100 árboles y random_state=42.
El modelo se guarda en `models/model.pkl`. La exactitud se calcula sobre el conjunto
de prueba. MLflow registra parámetros y métricas si está instalado y configurado.

## 9. Pruebas
Las pruebas se ejecutan con:
```bash
python -m pytest tests/ -v
```
Deben ejecutarse después de generar los CSV y el modelo.

## 10. Reproducibilidad y consideraciones
Se fija random_state=42 para la partición y el modelo. Los rangos del contrato son
reglas definidas para esta actividad y deben revisarse si cambia la fuente de datos.
Para migrar a ejecución productiva, se recomienda fijar versiones exactas de dependencias,
versionar datos y modelos, y automatizar pruebas en integración continua.
'''

readme_path = PROJECT_DIR / "README.md"
readme_path.write_text(readme_content, encoding="utf-8")
print("README creado:", readme_path)

README creado: C:\Users\Vivobook\OneDrive\Desktop\especializacion en ciencia de datos\seminario II\Proyecto II\README.md


## 10. Archivo de dependencias y configuración opcional de DVC

Se generan `requirements.txt` y un `dvc.yaml` básico. DVC es complementario: no es necesario para ejecutar `run_pipeline()` en Jupyter. Para utilizar DVC desde la terminal, debe estar instalado y el repositorio debe inicializarse con `dvc init`.

In [31]:
requirements_content = 'pandas\nscikit-learn\njoblib\nmlflow\npytest\njupyter\ndvc\n'

(PROJECT_DIR / "requirements.txt").write_text(requirements_content, encoding="utf-8")

dvc_yaml = 'stages:\n  prepare:\n    cmd: python src/prepare.py\n    deps:\n      - src/prepare.py\n    outs:\n      - data/processed/train.csv\n      - data/processed/test.csv\n  train:\n    cmd: python src/train.py\n    deps:\n      - src/train.py\n      - data/processed/train.csv\n      - data/processed/test.csv\n    outs:\n      - models/model.pkl\n'

(PROJECT_DIR / "dvc.yaml").write_text(dvc_yaml, encoding="utf-8")

print("requirements.txt y dvc.yaml creados.")
print("\nNota: dvc.yaml requiere que existan los scripts src/prepare.py y src/train.py.")
print("El pipeline de este notebook se ejecuta mediante run_pipeline(), no mediante dvc repro.")

requirements.txt y dvc.yaml creados.

Nota: dvc.yaml requiere que existan los scripts src/prepare.py y src/train.py.
El pipeline de este notebook se ejecuta mediante run_pipeline(), no mediante dvc repro.


## 11. Verificación final de los artefactos

Esta celda muestra los principales archivos que deben existir al terminar la actividad.

In [32]:
expected_files = [
    DATA_DIR / "train.csv",
    DATA_DIR / "test.csv",
    MODELS_DIR / "model.pkl",
    CONTRACTS_DIR / "data_contract.json",
    TESTS_DIR / "test_model.py",
    PROJECT_DIR / "README.md",
    PROJECT_DIR / "requirements.txt",
    PROJECT_DIR / "dvc.yaml",
]

for path in expected_files:
    status = "OK" if path.exists() else "PENDIENTE"
    print(f"[{status}] {path.relative_to(PROJECT_DIR)}")

[OK] data\processed\train.csv
[OK] data\processed\test.csv
[OK] models\model.pkl
[OK] contracts\data_contract.json
[OK] tests\test_model.py
[OK] README.md
[OK] requirements.txt
[OK] dvc.yaml


## 12. Conclusiones y observaciones generales

El notebook reúne las etapas principales de un flujo MLOps: preparación, validación, entrenamiento, pruebas y documentación. La orquestación se realiza mediante una única función de Python, lo que resulta apropiado para trabajar dentro de Jupyter.

### Observaciones técnicas
1. **Rutas:** Jupyter no define `__file__` en las celdas. Por eso se usa `Path.cwd()` y se configura `PROJECT_DIR`.
2. **Orden de ejecución:** ejecuta las celdas en orden. Si reinicias el kernel, debes volver a ejecutar las definiciones antes de llamar a `run_pipeline()`.
3. **Validación:** se implementó sin librerías externas especializadas; utiliza únicamente `csv`, `math` y `json` de Python.
4. **Inferencia:** el contrato define cuatro variables de entrada. `species` se reserva como etiqueta de entrenamiento y evaluación, no como dato requerido para predecir.
5. **MLflow:** su registro es complementario al guardado local del modelo. Si ocurre un problema de MLflow, se informa una advertencia y se conserva el artefacto local.
6. **DVC:** el `dvc.yaml` es una configuración de referencia para una migración a scripts `.py`. No ejecutará directamente las funciones que viven dentro del notebook.
7. **Reproducibilidad:** se fijan semillas aleatorias, pero para una reproducción estricta también se recomienda fijar las versiones exactas de las dependencias.

### Conclusión
La actividad permite pasar de una ejecución manual y fragmentada a un proceso organizado y repetible. La automatización reduce pasos manuales, las validaciones detectan datos que no cumplen las reglas y el contrato JSON establece una referencia para futuras integraciones de inferencia. La documentación facilita que otra persona configure y ejecute el proyecto.